# Train and evaluate a tiny adapter

Start with offline checks for response-only labels, low-rank updates, preference loss and release gates. The final bonus lab uses PyTorch on a synthetic binary routing task to make the frozen base and training updates observable.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Mask prompt and padding positions

Mask targets outside the assistant completion.

Given a token list, prompt length and real unpadded length, preserve exactly the assistant target span. Use -100 for prompt and padding labels and reject invalid boundaries.


In [ ]:
def response_labels(tokens, prompt_length, real_length):
    raise NotImplementedError


In [ ]:
assert response_labels([1,2,3,4,0,0],2,4) == [-100,-100,3,4,-100,-100]
assert response_labels([1,2],0,2) == [1,2]
assert response_labels([1,2],2,2) == [-100,-100]
try:
    response_labels([1],1,2)
except ValueError:
    pass
else:
    raise AssertionError("Reject spans outside the input")
for args in [([1,2,3],3,1),([1,2],-1,2)]:
    try:
        response_labels(*args)
    except ValueError:
        pass
    else:
        raise AssertionError("Reject a negative prompt length or a prompt longer than the real length")
print("OK")


## 2. Add a low-rank update

Compute a LoRA update with explicit factor sizes.

Implement add_adapter(base, ax, b, alpha). Return base plus alpha/r times B times ax, with r = len(ax). Reject zero rank, different output dimensions and B rows with the wrong width.


In [ ]:
def add_adapter(base, ax, b, alpha):
    raise NotImplementedError


In [ ]:
assert add_adapter([2,3],[4],[[0.5],[-1]],2) == [6.0,-5.0]
assert add_adapter([1],[2,3],[[4,5]],1) == [12.5]
assert add_adapter([5],[8],[[0]],4) == [5.0]
for base,ax,b in [([1],[],[[]]),([1,2],[1],[[1]]),([1],[2,3],[[1]])]:
    try:
        add_adapter(base,ax,b,1)
    except ValueError:
        pass
    else:
        raise AssertionError("Reject incompatible sizes")
print("OK")


## 3. Compute a stable DPO loss

Score a chosen/rejected response pair against a frozen reference.

Implement preference_loss(chosen, rejected, ref_chosen, ref_rejected, beta). Compute softplus of the negative reference-adjusted margin. Inputs are finite log probabilities and a positive beta, and tests include large margins.


In [ ]:
def preference_loss(chosen, rejected, ref_chosen, ref_rejected, beta):
    raise NotImplementedError


In [ ]:
from math import log
assert abs(preference_loss(-3,-4,-2,-3,1) - log(2)) < 1e-10
assert preference_loss(-2,-4,-3,-4,1) < log(2)
assert preference_loss(-4,-2,-4,-3,1) > log(2)
assert abs(preference_loss(-1000,0,0,0,1) - 1000) < 1e-8
assert preference_loss(0,-1000,0,0,1) < 1e-8
assert abs(preference_loss(-2,-4,-3,-4,0.5) - 0.4740769841801067) < 1e-9, "Apply beta to the margin"
print("OK")


## 4. Apply accuracy and critical-case gates

Apply a predefined holdout and critical-case release gate.

Implement release_gate(before, after, critical, min_gain). Return (absolute success-rate gain, approved). Reject nonmatching or empty binary outcomes. Require gain at least min_gain and no before-pass to after-fail on a critical row.


In [ ]:
def release_gate(before, after, critical, min_gain):
    raise NotImplementedError


In [ ]:
assert release_gate([1,0],[1,1],[True,False],0.5) == (0.5,True)
assert release_gate([1,0,0,0],[0,1,1,1],[True,False,False,False],0.25) == (0.5,False)
assert release_gate([0,1],[0,1],[False,True],0.01) == (0.0,False)
assert release_gate([0,0],[0,1],[True,False],0.5) == (0.5,True), "A critical row that already failed before is not a regression"
try:
    release_gate([],[],[],0)
except ValueError:
    pass
else:
    raise AssertionError("Empty eval should not approve a release")
print("OK")


## 5. Track preference loss and holdout release evidence separately

Implement tuning_report(train_pairs, before, after, critical, beta, min_gain). Compute average
preference_loss over the nonempty sequence of training pairs (chosen, rejected, reference chosen,
reference rejected). Use release_gate on the aligned holdout outcomes. Return (mean train pair loss,
holdout gain, approved). Reject an empty training list. Training loss must not affect approval.


In [ ]:
def tuning_report(train_pairs, before, after, critical, beta, min_gain):
    raise NotImplementedError


In [ ]:
from math import log
result = tuning_report([(-1,-2,-1,-2),(-8,-9,-2,-3)],[0,1],[1,1],[False,True],1,0.5)
assert abs(result[0] - log(2)) < 1e-9 and result[1:] == (0.5,True)
result = tuning_report([(-1,-20,-2,-3)],[1,0,0],[0,1,1],[True,False,False],1,0.2)
assert result[0] < 1e-6 and result[2] is False
try:
    tuning_report([], [0], [1], [False], 1, 0)
except ValueError:
    pass
else:
    raise AssertionError("Reject an empty training set")
print("OK")


## Interpretation

Record when you would use prompt changes, RAG, supervised LoRA or preference tuning. Name a critical holdout case and explain how you would gather independent evidence for it. Record the base checkpoint, tokenizer/template revision, target modules, rank and alpha you would version.

**Your answer**


## Bonus lab: train the adapter on a one-token task

Run this cell in Colab with PyTorch available. CPU is sufficient and no model download is needed.
Two input features stand in for a frozen model's representation. A rank-one adapter first learns
a target token by cross-entropy, then learns from chosen/rejected alternatives relative to a
frozen SFT snapshot. A classifier over two possible next tokens makes the exact training loop
visible. A real decoder would score every unmasked token of each completion.

Record the base, SFT and DPO accuracy and the initial/final preference losses. Try changing beta
without touching the holdout. The two held-out cases illustrate the split, but are too few to
justify deployment.


In [ ]:
import importlib.util

if importlib.util.find_spec("torch") is None:
    print("Optional lab skipped. Run this cell in a Colab notebook with PyTorch.")
else:
    import torch
    from torch import nn
    from torch.nn import functional as F

    # One-token completion task: 0 = hours, 1 = catalogue.
    # The two inputs are signed intent evidence and a distractor.
    train_x = torch.tensor([[-2., 0.], [-1., 1.], [1., -1.], [2., 1.]])
    train_y = torch.tensor([0, 0, 1, 1])
    holdout_x = torch.tensor([[-1.5, 2.], [1.5, -2.]])
    holdout_y = torch.tensor([0, 1])

    class TinyAdapter(nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer("base", torch.zeros(2, 2))
            self.a = nn.Parameter(torch.tensor([[0.25, 0.0]]))
            self.b = nn.Parameter(torch.zeros(2, 1))

        def forward(self, x):
            # Rank and alpha are both one, so alpha/r = 1.
            return F.linear(x, self.base) + (x @ self.a.T) @ self.b.T

    adapter = TinyAdapter()
    original_base = adapter.base.clone()
    with torch.no_grad():
        base_scores = F.linear(holdout_x, adapter.base)
        assert torch.equal(adapter(holdout_x), base_scores)
        base_accuracy = (base_scores.argmax(-1) == holdout_y).float().mean().item()

    optimizer = torch.optim.SGD(adapter.parameters(), lr=0.5)
    for _ in range(40):
        optimizer.zero_grad()
        sft_loss = F.cross_entropy(adapter(train_x), train_y)
        sft_loss.backward()
        optimizer.step()

    # Hold the SFT model fixed as the reference for the preference stage.
    # Each rejected token is the other class for the same input.
    with torch.no_grad():
        reference = adapter(train_x).log_softmax(dim=-1)
        sft_accuracy = (adapter(holdout_x).argmax(-1) == holdout_y).float().mean().item()

    def chosen_minus_rejected(log_probs):
        chosen = log_probs.gather(1, train_y[:, None]).squeeze(1)
        rejected = log_probs.gather(1, (1 - train_y)[:, None]).squeeze(1)
        return chosen - rejected

    reference_margin = chosen_minus_rejected(reference)
    beta = 0.5

    def dpo_loss():
        policy_margin = chosen_minus_rejected(adapter(train_x).log_softmax(dim=-1))
        return F.softplus(-beta * (policy_margin - reference_margin)).mean()

    first_loss = dpo_loss().item()
    for _ in range(30):
        optimizer.zero_grad()
        loss = dpo_loss()
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        last_loss = dpo_loss().item()
        final_accuracy = (adapter(holdout_x).argmax(-1) == holdout_y).float().mean().item()

    assert torch.equal(adapter.base, original_base), "The base must stay frozen"
    assert last_loss < first_loss, "The training preference objective should fall"
    assert final_accuracy >= base_accuracy, "Check that the toy holdout has not regressed"
    print(f"Base / SFT / DPO holdout accuracy: {base_accuracy:.2f} / {sft_accuracy:.2f} / {final_accuracy:.2f}")
    print(f"Training preference loss: {first_loss:.3f} -> {last_loss:.3f}")
